In [2]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
import sys

from utils import scrollview
from pedsilicoICH.ground_truth_definition.phantoms import *
from pedsilicoICH.image_reconstruction import *

df_volume = pd.read_csv('../src/pedsilicoICH/distributions/BHSD_volume_distributions.csv')
df_HU = pd.read_csv('../src/pedsilicoICH/distributions/BHSD_HU_distributions.csv')

mida_path = Path('/home/jayse.weaver/the_lab/MIDA_v1.0/MIDA_v1_voxels')
NIHPD_path = Path('/home/jayse.weaver/PedSilicoICH/NIHPD_Head_Phantom/')
#phantom = MIDA_Head(mida_path)

# TEST PERFORMANCE
#volumes = np.linspace(1, 100, 100)
volumes = [25]
new_volumes = []

for i in range(len(volumes)):
    print(i)
    phantom = MIDA_Head(mida_path)

    dx = phantom.dx
    dy = phantom.dy
    dz = phantom.dz

    # optional, add lesion (comment out if just testing recon)
    phantom.insert_lesion(lesion_type='sphere', volume=volumes[i], contrast=100,
                        init_slice=None, mass_effect=False, seed=None)

    hemorrhage_volume = (len(np.argwhere(phantom._lesion[0] != 0)))*((dx*dy*dz)/1000)
    print('new volume: ' + str(hemorrhage_volume))
    new_volumes.append(hemorrhage_volume)

dfsyn = pd.DataFrame()
dfsyn['defined'] = volumes
dfsyn['new'] = new_volumes

scrollview(phantom._phantom)

nib.save(nib.Nifti1Image(phantom._phantom, np.eye(4)), 'test.nii')

scrollview(phantom._lesion[0])


0


/home/jayse.weaver/PedSilicoICH/src/pedsilicoICH/ground_truth_definition/phantoms.py:358: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  material_lut.loc[material_lut.material == 'Cerebellum  Gray Matter', 'xcist material'] = 'gray_matter'
/home/jayse.weaver/PedSilicoICH/src/pedsilicoICH/ground_truth_definition/phantoms.py:359: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  material_lut.loc[material_lut.material == 'Cerebellum  Gray Matter', 'CT Number [HU]'] = self.gm_HU


saving dataframe
saving dataframe
new volume: 25.029625


interactive(children=(IntSlider(value=240, description='idx', max=479), Output()), _dom_classes=('widget-inter…

interactive(children=(IntSlider(value=240, description='idx', max=479), Output()), _dom_classes=('widget-inter…

In [ ]:
# temp
import math

slice_coverage = []
for i in range(len(volumes)):
    #slice_coverage.append(13.942*math.log(volumes[i]) + 13.449)
    slice_coverage.append(17.739*math.log(volumes[i]) + 17.314)

plt.figure()
plt.plot(volumes, slice_coverage)
plt.xlabel('Input Volume')
plt.ylabel('Slice Coverage')
plt.plot(new_volumes, slice_coverage[:len(new_volumes)])

plt.figure()
plt.plot(volumes, volumes)
plt.plot(volumes[:len(new_volumes)], new_volumes)
plt.xlabel('Input Volume')
plt.ylabel('Output Volume')

In [ ]:
from pedsilicoICH.image_acquisition import Scanner

phantom.patient_name = 'test_70kv_mono'
scanner = Scanner(phantom)
scanner

In [ ]:
scanner.scout_view(startZ=10, endZ=18, table_speed=0)

In [ ]:
scanner.run_scan(mA=280, kVp=70, startZ=10, endZ=18, views=1000)

In [ ]:
scanner.run_recon(sliceThickness=5, kernel='standard')

In [ ]:
scanner.write_to_dicom("TEST_NIHPD_70kv_mono_standard1000.dcm")

scrollview(scanner.recon)